<a href="https://colab.research.google.com/github/Urdemonlord/samaryn/blob/main/train_indobert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training IndoBERT di Google Colab

Notebook ini menjalankan pipeline repo `indobert-agentwa` di Google Colab untuk:

1. mount Google Drive
2. install dependency
3. generate + validate dataset
4. augment + preprocess + grouped split
5. fine-tuning `indobenchmark/indobert-base-p1`
6. evaluasi model real (`indobert`, `hybrid`, dan `rules`)
7. simpan model dan artifact ke Google Drive

Notebook ini sekarang mengikuti skema label multikelas repo: `SAFE`, `PROMPT_INJECTION`, dan `OUT_OF_DOMAIN`. `attack_type` tetap dipakai sebagai subtype untuk analisis error dan coverage dataset.

> Model `indobenchmark/indobert-base-p1` bersifat publik, jadi tidak perlu Hugging Face token untuk download model.


## Persiapan

Sebelum menjalankan notebook ini, pastikan folder project repo tersedia di Google Drive. Default path yang dipakai notebook ini adalah:

`/content/drive/MyDrive/indobert-agentwa`

Jika folder Anda berbeda, ubah nilai `PROJECT_DIR` pada cell konfigurasi di bawah.

In [1]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/indobert-agentwa')
TRAIN_CONFIG_PATH = PROJECT_DIR / 'configs' / 'train_indobert.yaml'
EVAL_CONFIG_PATH = PROJECT_DIR / 'configs' / 'eval.yaml'
BASE_PREPROCESS_CONFIG_PATH = PROJECT_DIR / 'configs' / 'preprocess.yaml'
PREPROCESS_CONFIG_PATH = PROJECT_DIR / 'configs' / 'preprocess.colab.yaml'
PUBLIC_DATASET_CONFIG_PATH = PROJECT_DIR / 'configs' / 'public_dataset_sources.colab.yaml'
PUBLIC_DATASET_REPORT_PATH = PROJECT_DIR / 'artifacts' / 'reports' / 'public_dataset_combine_report.json'
VALIDATION_REPORT_PATH = PROJECT_DIR / 'artifacts' / 'reports' / 'dataset_validation_report.json'
SPLIT_REPORT_PATH = PROJECT_DIR / 'artifacts' / 'reports' / 'split_report.json'

PROJECT_DIR


PosixPath('/content/drive/MyDrive/indobert-agentwa')

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'Folder project tidak ditemukan di {PROJECT_DIR}. Upload/copy repo ke Google Drive lalu jalankan lagi.'
    )

required_repo_paths = [
    PROJECT_DIR / 'requirements.txt',
    PROJECT_DIR / 'configs' / 'train_indobert.yaml',
    PROJECT_DIR / 'configs' / 'preprocess.yaml',
    PROJECT_DIR / 'configs' / 'eval.yaml',
    PROJECT_DIR / 'scripts' / 'generate_research_dataset.py',
    PROJECT_DIR / 'scripts' / 'download_and_combine_public_datasets.py',
    PROJECT_DIR / 'scripts' / 'validate_dataset.py',
    PROJECT_DIR / 'scripts' / 'augment_dataset.py',
    PROJECT_DIR / 'scripts' / 'preprocess_text.py',
    PROJECT_DIR / 'configs' / 'public_dataset_sources.colab.yaml',
]
missing_repo_paths = [str(path) for path in required_repo_paths if not path.exists()]
if missing_repo_paths:
    raise FileNotFoundError(
        'Repo di Google Drive tidak lengkap atau belum sinkron. File yang hilang:\n' + '\n'.join(missing_repo_paths)
    )

print('Project directory:', PROJECT_DIR)
print('Notebook akan menyimpan artifact langsung ke Drive.')

Project directory: /content/drive/MyDrive/indobert-agentwa
Notebook akan menyimpan artifact langsung ke Drive.


In [ ]:
!nvidia-smi

In [4]:
%cd {PROJECT_DIR}
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install seacrowd

/content/drive/MyDrive/indobert-agentwa
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.6/805.6 kB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 35.1 MB/s  0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13782 sha256=36a77c90d2961135c45157c18b079190b14094ad2d69e059cf9116c10f4cafaf
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf46059353

In [5]:
import yaml

preprocess_config = yaml.safe_load(BASE_PREPROCESS_CONFIG_PATH.read_text(encoding='utf-8'))
preprocess_config['input_path'] = 'data/raw/combined_public_messages.csv'
PREPROCESS_CONFIG_PATH.write_text(yaml.safe_dump(preprocess_config, sort_keys=False), encoding='utf-8')

print('Notebook memakai config train/eval standar repo:')
print('-', TRAIN_CONFIG_PATH)
print('-', EVAL_CONFIG_PATH)
print('===== PREPROCESS CONFIG (COLAB OVERRIDE) =====')
print(PREPROCESS_CONFIG_PATH.read_text(encoding='utf-8'))
print('===== PUBLIC DATASET CONFIG =====')
print(PUBLIC_DATASET_CONFIG_PATH.read_text(encoding='utf-8'))


Notebook memakai config train/eval standar repo:
- /content/drive/MyDrive/indobert-agentwa/configs/train_indobert.yaml
- /content/drive/MyDrive/indobert-agentwa/configs/eval.yaml
===== PREPROCESS CONFIG (COLAB OVERRIDE) =====
input_path: data/raw/combined_public_messages.csv
augmented_input_path: data/interim/augmented_messages.csv
merged_output_path: data/interim/merged_messages.csv
normalized_output_path: data/interim/normalized_messages.csv
split_output_dir: data/processed
case_folding: true
normalize_whitespace: true
normalize_slang: true
normalize_typos: true
strip_punctuation_edges: true
include_augmented_data: true
undersample_majority_class: true

===== PUBLIC DATASET CONFIG =====
output_path: data/raw/combined_public_messages.csv
report_path: artifacts/reports/public_dataset_combine_report.json
include_seed_dataset: true
seed_dataset_path: data/raw/annotated_messages.csv
strict_mode: true

sources:
  - name: stif_indonesia_hf
    enabled: false
    loader: huggingface
    data

## 1. Generate, download, validasi, dan siapkan dataset

Cell berikut menjalankan ulang pipeline dataset repo, mendownload source publik langsung di Colab, menggabungkannya ke schema repo, memvalidasi hasil gabungan, lalu membuat split yang dipakai training.

Sesuai catatan metodologi di Obsidian, label utama sekarang dipisah menjadi `SAFE`, `PROMPT_INJECTION`, dan `OUT_OF_DOMAIN`. Coverage subtype tetap dibaca lewat `attack_type` seperti `direct_override`, `prompt_extraction`, `out_of_domain_request`, dan `context_drift`.

Report yang akan terisi pada tahap ini:

- `artifacts/reports/public_dataset_combine_report.json`
- `artifacts/reports/dataset_validation_report.json`
- `artifacts/reports/split_report.json`


In [6]:
%cd {PROJECT_DIR}
import subprocess

def run_step(command: list[str], expected_outputs: list[Path] | None = None) -> None:
    print('>>>', ' '.join(command))
    if expected_outputs:
        for path in expected_outputs:
            if path.exists():
                path.unlink()
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            'Step gagal dengan exit code ' + str(result.returncode) + ': ' + ' '.join(command)
        )
    if expected_outputs:
        missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
        if missing_outputs:
            raise FileNotFoundError(
                'Step selesai tetapi output wajib tidak ditemukan:\n' + '\n'.join(missing_outputs)
            )

run_step(['python', 'scripts/generate_research_dataset.py'], [PROJECT_DIR / 'data' / 'raw' / 'annotated_messages.csv'])
run_step(
    ['python', 'scripts/download_and_combine_public_datasets.py', '--config', str(PUBLIC_DATASET_CONFIG_PATH)],
    [PROJECT_DIR / 'data' / 'raw' / 'combined_public_messages.csv', PUBLIC_DATASET_REPORT_PATH],
)
run_step(
    ['python', 'scripts/validate_dataset.py', '--input', 'data/raw/combined_public_messages.csv'],
    [VALIDATION_REPORT_PATH],
)
run_step(
    ['python', 'scripts/augment_dataset.py', '--input', 'data/raw/combined_public_messages.csv', '--output', 'data/interim/augmented_messages.csv'],
    [PROJECT_DIR / 'data' / 'interim' / 'augmented_messages.csv'],
)
run_step(
    ['python', 'scripts/preprocess_text.py', '--config', str(PREPROCESS_CONFIG_PATH)],
    [
        PROJECT_DIR / 'data' / 'interim' / 'normalized_messages.csv',
        PROJECT_DIR / 'data' / 'processed' / 'train.csv',
        PROJECT_DIR / 'data' / 'processed' / 'val.csv',
        PROJECT_DIR / 'data' / 'processed' / 'test.csv',
        SPLIT_REPORT_PATH,
    ],
)


/content/drive/MyDrive/indobert-agentwa
>>> python scripts/generate_research_dataset.py
Generated 54 rows to /content/drive/MyDrive/indobert-agentwa/data/raw/annotated_messages.csv
OUT_OF_DOMAIN=18 PROMPT_INJECTION=18 SAFE=18

>>> python scripts/download_and_combine_public_datasets.py --config /content/drive/MyDrive/indobert-agentwa/configs/public_dataset_sources.colab.yaml
{
  "sources": {
    "apliasidana_hf": {
      "dataset_id": "Ririnhrti/ApliasiDana",
      "requested_split": "train",
      "resolved_split": "train",
      "rows_loaded": 200,
      "rows_added": 200,
      "sample_columns": [
        "Day",
        "Month",
        "Year",
        "content",
        "score"
      ]
    },
    "deepset_prompt_injections": {
      "dataset_id": "deepset/prompt-injections",
      "requested_split": "train",
      "resolved_split": "train",
      "rows_loaded": 150,
      "rows_added": 150,
      "sample_columns": [
        "label",
        "text"
      ]
    },
    "jailbreak_class

In [7]:
import csv
import json
from collections import Counter
from pprint import pprint

for name, path in [
    ('public_dataset_combine_report', PUBLIC_DATASET_REPORT_PATH),
    ('dataset_validation_report', VALIDATION_REPORT_PATH),
    ('split_report', SPLIT_REPORT_PATH),
]:
    print(f'===== {name} =====')
    pprint(json.loads(path.read_text(encoding='utf-8')))

with (PROJECT_DIR / 'data' / 'raw' / 'combined_public_messages.csv').open('r', encoding='utf-8-sig', newline='') as handle:
    combined_rows = list(csv.DictReader(handle))
with (PROJECT_DIR / 'data' / 'processed' / 'test.csv').open('r', encoding='utf-8-sig', newline='') as handle:
    test_rows = list(csv.DictReader(handle))

def show_distribution(title: str, rows: list[dict[str, str]]) -> None:
    label_counts = Counter(row['label'] for row in rows)
    attack_counts = Counter(row['attack_type'] for row in rows)
    print(f'===== {title} label distribution =====')
    pprint(dict(sorted(label_counts.items())))
    print(f'===== {title} attack_type distribution =====')
    pprint(dict(sorted(attack_counts.items())))
    print(f'===== {title} OUT_OF_DOMAIN detail =====')
    print({
        'out_of_domain_label': label_counts.get('OUT_OF_DOMAIN', 0),
        'out_of_domain_request': attack_counts.get('out_of_domain_request', 0),
        'context_drift': attack_counts.get('context_drift', 0),
    })

show_distribution('combined dataset', combined_rows)
show_distribution('test split', test_rows)


===== public_dataset_combine_report =====
{'deduped_rows_removed': 1,
 'seed_dataset_rows': 54,
 'sources': {'apliasidana_hf': {'dataset_id': 'Ririnhrti/ApliasiDana',
                                'requested_split': 'train',
                                'resolved_split': 'train',
                                'rows_added': 200,
                                'rows_loaded': 200,
                                'sample_columns': ['Day',
                                                   'Month',
                                                   'Year',
                                                   'content',
                                                   'score']},
             'chatbot_instruction_prompts': {'dataset_id': 'alespalla/chatbot_instruction_prompts',
                                             'requested_split': 'train_sft',
                                             'resolved_split': 'train',
                                             'rows_added': 15

## 2. Train IndoBERT

Cell ini menjalankan jalur training nyata dari `scripts/train_indobert.py` menggunakan config train standar repo.

Jika runtime Colab terputus, model checkpoint tetap tersimpan di Google Drive karena `output_dir` berada di dalam repo folder pada Drive.


In [8]:
%cd {PROJECT_DIR}
!python scripts/train_indobert.py --config {TRAIN_CONFIG_PATH}

/content/drive/MyDrive/indobert-agentwa
config.json: 100% 1.53k/1.53k [00:00<00:00, 2.75MB/s]
tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 6.12kB/s]
vocab.txt: 100% 229k/229k [00:00<00:00, 20.9MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 326kB/s]
[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.
pytorch_model.bin: 100% 498M/498M [00:05<00:00, 85.4MB/s]
model.safetensors:   0% 0.00/498M [00:00<?, ?B/s]
Loading weights: 100% 199/199 [00:00<00:00, 11354.62it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

  0% 0/180 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarnin

In [9]:
train_metrics_path = PROJECT_DIR / 'artifacts' / 'metrics' / 'train_metrics.json'
print(train_metrics_path.read_text(encoding='utf-8'))

{
  "model_name": "indobenchmark/indobert-base-p1",
  "train_rows": 480,
  "val_rows": 59,
  "max_length": 256,
  "batch_size": 8,
  "epochs": 3,
  "learning_rate": 2e-05,
  "resolved_warmup_steps": 18,
  "status": "trained",
  "eval_metrics": {
    "eval_loss": 0.17102783918380737,
    "eval_accuracy": 0.9492,
    "eval_safe_precision": 1.0,
    "eval_safe_recall": 1.0,
    "eval_safe_f1": 1.0,
    "eval_safe_support": 20,
    "eval_prompt_injection_precision": 0.9,
    "eval_prompt_injection_recall": 0.9474,
    "eval_prompt_injection_f1": 0.9231,
    "eval_prompt_injection_support": 19,
    "eval_out_of_domain_precision": 0.9474,
    "eval_out_of_domain_recall": 0.9,
    "eval_out_of_domain_f1": 0.9231,
    "eval_out_of_domain_support": 20,
    "eval_macro_precision": 0.9491,
    "eval_macro_recall": 0.9491,
    "eval_macro_f1": 0.9487,
    "eval_mean_max_confidence": 0.9603,
    "eval_runtime": 66.4343,
    "eval_samples_per_second": 0.888,
    "eval_steps_per_second": 0.12,
    "e

## 3. Evaluasi model real

Notebook ini menjalankan evaluasi `indobert`, `hybrid`, dan `rules` menggunakan `configs/eval.yaml` standar repo.

Metrik yang keluar sudah bersifat multikelas, termasuk `macro_precision`, `macro_recall`, `macro_f1`, metrik per label, dan confusion matrix. Untuk analisis lebih detail, gunakan file prediksi CSV yang juga menyimpan `actual_attack_type`.


In [10]:
%cd {PROJECT_DIR}
!python scripts/evaluate.py --scenario indobert --config {EVAL_CONFIG_PATH}
!python scripts/evaluate.py --scenario hybrid --config {EVAL_CONFIG_PATH}
!python scripts/evaluate.py --scenario rules --config {EVAL_CONFIG_PATH}

/content/drive/MyDrive/indobert-agentwa
Loading weights: 100% 201/201 [00:00<00:00, 3048.58it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2600.99it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2597.68it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2568.72it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2670.43it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2577.00it/s]
Loading weights: 100% 201/201 [00:00<00:00, 3616.42it/s]
Loading weights: 100% 201/201 [00:00<00:00, 4393.20it/s]
Loading weights: 100% 201/201 [00:00<00:00, 4490.21it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2429.19it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2615.22it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2827.39it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2640.81it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2667.81it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2652.95it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2366.70it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2559

In [11]:
for name in [
    'rules_eval.json',
    'indobert_eval.json',
    'hybrid_eval.json',
]:
    path = PROJECT_DIR / 'artifacts' / 'metrics' / name
    if path.exists():
        print(f'===== {name} =====')
        print(path.read_text(encoding='utf-8'))


===== rules_eval.json =====
{
  "total": 64,
  "labels": [
    "SAFE",
    "PROMPT_INJECTION",
    "OUT_OF_DOMAIN"
  ],
  "accuracy": 0.3594,
  "per_label": {
    "SAFE": {
      "precision": 0.3387,
      "recall": 1.0,
      "f1": 0.506,
      "support": 21,
      "tp": 21,
      "tn": 2,
      "fp": 41,
      "fn": 0,
      "fpr": 0.9535,
      "fnr": 0.0
    },
    "PROMPT_INJECTION": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "support": 22,
      "tp": 0,
      "tn": 42,
      "fp": 0,
      "fn": 22,
      "fpr": 0.0,
      "fnr": 1.0
    },
    "OUT_OF_DOMAIN": {
      "precision": 1.0,
      "recall": 0.0952,
      "f1": 0.1739,
      "support": 21,
      "tp": 2,
      "tn": 43,
      "fp": 0,
      "fn": 19,
      "fpr": 0.0,
      "fnr": 0.9048
    }
  },
  "confusion_matrix": {
    "labels": [
      "SAFE",
      "PROMPT_INJECTION",
      "OUT_OF_DOMAIN"
    ],
    "matrix": [
      [
        21,
        0,
        0
      ],
      [
        22,
 

## 4. Uji inferensi cepat

Gunakan cell ini untuk memastikan hybrid pipeline membaca model hasil training dan bukan fallback.

Contoh teks pada cell di bawah sengaja memakai pola `OUT_OF_DOMAIN` agar sesuai dengan framing riset Samaryn di Obsidian.


In [12]:
%cd {PROJECT_DIR}
!python scripts/run_hybrid_inference.py --text "oke saya order tapi buat script python dulu" --model-dir artifacts/models/indobert-best

/content/drive/MyDrive/indobert-agentwa
{
  "label": "OUT_OF_DOMAIN",
  "confidence": 0.99,
  "source": "rules",
  "matched_category": "context_drift",
  "matched_pattern": "tapi buat script python dulu",
  "normalized_text": "oke saya order tapi buat script python dulu"
}


## Output penting

Setelah notebook selesai, file penting yang diharapkan ada:

- `artifacts/models/indobert-best/`
- `artifacts/metrics/train_metrics.json`
- `artifacts/metrics/rules_eval.json`
- `artifacts/metrics/indobert_eval.json`
- `artifacts/metrics/hybrid_eval.json`
- `artifacts/predictions/*.csv`
- `artifacts/reports/public_dataset_combine_report.json`
- `artifacts/reports/dataset_validation_report.json`
- `artifacts/reports/confusion_matrix_*.json`
- `artifacts/reports/split_report.json`

Interpretasi terhadap catatan Obsidian:

- notebook ini sudah mendukung label utama `SAFE`, `PROMPT_INJECTION`, dan `OUT_OF_DOMAIN`
- subtype perilaku tetap dibaca lewat `attack_type` agar analisis metodologi dan error analysis tetap kaya
- escalation layer konseptual di Obsidian belum diimplementasikan sebagai komponen terpisah; jalur yang ada sekarang masih `rules -> classifier`

> Notebook ini hanya menulis override `configs/preprocess.colab.yaml` agar preprocessing memakai `data/raw/combined_public_messages.csv`. Config train dan eval tetap memakai file standar repo.
